# Baseline Evaluation — Before Fine-Tuning

## Goal
Before fine-tuning, we need to understand what the model already knows.
This is our baseline — the starting point we will compare against.

## What We Measure
- Text generation quality
- Perplexity on our target domain
- Sample outputs on domain-specific prompts

## Model
DistilGPT-2 — pre-trained on general web text (WebText dataset)

In [1]:
import torch
from transformers import GPT2Tokenizer, GPT2LMHeadModel
import numpy as np
import matplotlib.pyplot as plt

# Check device
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Device: {device}")
print(f"PyTorch version: {torch.__version__}")

Device: mps
PyTorch version: 2.12.0


## 1. Load Pre-trained Model

In [2]:
model_name = "distilgpt2"

print(f"Loading {model_name}...")
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
model = GPT2LMHeadModel.from_pretrained(model_name)
model = model = model.to(device)
model.eval()

# Add padding token
tokenizer.pad_token = tokenizer.eos_token

total_params = sum(p.numel() for p in model.parameters())
print(f"Model loaded.")
print(f"Total parameters: {total_params:,}")
print(f"Model size: ~{total_params * 4 / 1024**2:.0f} MB")

Loading distilgpt2...


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

error uploading: 'code'


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Model loaded.
Total parameters: 81,912,576
Model size: ~312 MB


## 2. Generate Text — Baseline
What does the model generate before fine-tuning?

In [6]:
def generate_text(prompt: str, max_new_tokens: int = 100) -> str:
    """Generate text from a prompt using the current model."""
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    
    with torch.no_grad():
        outputs = model.generate(
            inputs.input_ids,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.8,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id
        )
    
    generated = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return generated


# Test on domain-specific prompts
prompts = [
    "The transformer architecture uses attention to",
    "In machine learning, fine-tuning means",
    "Retrieval augmented generation works by",
]

print("=== BASELINE GENERATION ===\n")
for prompt in prompts:
    print(f"Prompt: {prompt}")
    output = generate_text(prompt)
    print(f"Output: {output}")
    print("-" * 60)

=== BASELINE GENERATION ===

Prompt: The transformer architecture uses attention to
Output: The transformer architecture uses attention to detail, the complexity of the structure, and the complexity of the material and the shape of the material. The current design requires a combination of two elements that can be integrated into the design, the structural design, and the structural design. This approach has been developed with a focus on the use of materials to develop a "super-structural" structure.




This article discusses the design of the building system.

The design of the building system is not intended to be a
------------------------------------------------------------
Prompt: In machine learning, fine-tuning means
Output: In machine learning, fine-tuning means building, and designing large and small businesses. And that's not to say that computer-learning is not going to be a big hit. But, as you might have already seen, the rise of machine learning is already taking place 

## 3. Measure Perplexity
Perplexity measures how "surprised" the model is by text.
Lower perplexity = model understands the text better.

In [8]:
def calculate_preplexity(text: str) -> float:
    """
    Calculate preplexity of a text under the current model.
    Lower = model understands this text better.
    """
    inputs = tokenizer(text, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model(**inputs, labels=inputs.input_ids)
        loss = outputs.loss

    return float(torch.exp(loss))


# Test on ML text vs general text
ml_texts = [
    "The transformer model uses self-attention mechanisms to process sequences in parallel.",
    "Fine-tuning adapts a pre-trained model to a specific domain using labeled data.",
    "BERT uses bidirectional attention to understand context from both directions.",    
]

general_texts = [
    "The weather today is sunny and warm with clear skies.",
    "I went to the grocery store to buy fruits and vegetables.",
    "The football team won the championship after a hard-fought season.",
]

print(f"{'Text':<60} {'Preplexity':>12}")
print("-" * 74)
print("ML Texts:")
ml_preplexities = []
for text in ml_texts:
    p = calculate_preplexity(text)
    ml_preplexities.append(p)
    print(f"  {text[:55]:<55} {p:>12.2f}")

print("\nGeneral Texts:")
general_preplexities = []
for text in general_texts:
    p = calculate_preplexity(text)
    general_preplexities.append(p)
    print(f"  {text[:55]:<55} {p:>12.2f}")

print(f"\nAvg ML preplexity:     {np.mean(ml_preplexities):.2f}")
print(f"Avg General preplexity: {np.mean(general_preplexities):.2f}")

Text                                                           Preplexity
--------------------------------------------------------------------------
ML Texts:
  The transformer model uses self-attention mechanisms to       310.22
  Fine-tuning adapts a pre-trained model to a specific do       154.37
  BERT uses bidirectional attention to understand context       150.43

General Texts:
  The weather today is sunny and warm with clear skies.          96.23
  I went to the grocery store to buy fruits and vegetable        19.39
  The football team won the championship after a hard-fou        30.07

Avg ML preplexity:     205.01
Avg General preplexity: 48.56


## 4. Key Observations

| Text Type | Avg Perplexity | Interpretation |
|-----------|---------------|----------------|
| ML/NLP text | 205.01 | Model struggles — domain mismatch |
| General text | 48.56 | Model understands well |

## Key Insight
DistilGPT-2 was trained on general web text.
ML terminology like "self-attention", "fine-tuning", "BERT" is rare in that corpus.
The model treats these as uncommon words — hence high perplexity.

## What Fine-Tuning Will Do
Train the model on ML text so it learns:
- ML vocabulary and patterns
- How technical sentences are structured
- Domain-specific relationships between concepts

## Goal
After fine-tuning, ML text perplexity should drop significantly.
-> 02_data_preparation.ipynb